<a href="https://colab.research.google.com/github/Sergiodzm99/Proyect-Model_MLB/blob/main/API_STATS_%2B_Baseball_Savant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color='00FF00'> Datos Validados



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### <font color='00FFFF'> Extrer API Stats
EL archivo se generó utilizando de los siguientes codigos:
* [API Stats a DF](https://colab.research.google.com/drive/1fOqxhmISdVA1b-LvCHMmqAl62K0kZg4n?authuser=1)


In [ ]:
df_stats = pd.read_parquet('/content/drive/MyDrive/1. Cursos/1. UNIR/Master IA/4to Semestre/Proyecto Final/Base de Datos/Datos API STATS/df_apistats.parquet')

### <font color='00FFFF'> Extrer SAVANT
EL archivo se generó utilizando de los siguientes codigos:
* [BASE cruda SAVANT](https://colab.research.google.com/drive/1DSnlJHeEuPxSQEqnA0KrgVl240QUmmJH?authuser=1)
* [Conversión y limpieza a DF](https://colab.research.google.com/drive/1-m54AMmlYXFbVqBaS3aot_k9WRAU0Nkr?authuser=1#scrollTo=AKtYma_4Hnc4)

In [ ]:
df_savant = pd.read_parquet('/content/drive/MyDrive/1. Cursos/1. UNIR/Master IA/4to Semestre/Proyecto Final/Base de Datos/Datos SAVAT/df_savant.parquet')

### <font color='00FFFF'> Validar Primary Key para Merge / Join

In [ ]:
team_map = {
    "AZ": "Arizona Diamondbacks",
    "ATL": "Atlanta Braves",
    "BAL": "Baltimore Orioles",
    "BOS": "Boston Red Sox",
    "CHC": "Chicago Cubs",
    "CWS": "Chicago White Sox",
    "CIN": "Cincinnati Reds",
    "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies",
    "DET": "Detroit Tigers",
    "HOU": "Houston Astros",
    "KC": "Kansas City Royals",
    "LAA": "Los Angeles Angels",
    "LAD": "Los Angeles Dodgers",
    "MIA": "Miami Marlins",
    "MIL": "Milwaukee Brewers",
    "MIN": "Minnesota Twins",
    "NYM": "New York Mets",
    "NYY": "New York Yankees",
    "ATH": "Athletics",
    "PHI": "Philadelphia Phillies",
    "PIT": "Pittsburgh Pirates",
    "SD": "San Diego Padres",
    "SEA": "Seattle Mariners",
    "SF": "San Francisco Giants",
    "STL": "St. Louis Cardinals",
    "TB": "Tampa Bay Rays",
    "TEX": "Texas Rangers",
    "TOR": "Toronto Blue Jays",
    "WSH": "Washington Nationals"
}

In [ ]:
df_savant["team_name"] = df_savant["team"].map(team_map)

In [ ]:
df_stats["home_team_name"] = df_stats["teams.home.team.name"].replace({
    "Oakland Athletics": "Athletics"
})

df_stats["away_team_name"] = df_stats["teams.away.team.name"].replace({
    "Oakland Athletics": "Athletics"
})

In [ ]:
set_savant = set(df_savant["team_name"].unique())

set_stats = (
    set(df_stats["home_team_name"].unique())
    | set(df_stats["away_team_name"].unique())
)

set_stats - set_savant

set()

### <font color='00FFFF'> Merge

Preparar dos versiones de df_savant:

In [ ]:
rolling_cols = [
    c for c in df_savant.columns
    if c.endswith("_last15")
]

savant_home = df_savant[
    ["game_pk", "team_name"] + rolling_cols
].copy()

savant_home = savant_home.rename(
    columns={col: f"home_{col.replace('team_avg_', '').replace('_game_last15', '_last15')}"
             for col in rolling_cols}
)

savant_home = savant_home.rename(columns={
    "team_name": "home_team_name"
})

In [ ]:
savant_away = df_savant[
    ["game_pk", "team_name"] + rolling_cols
].copy()

savant_away = savant_away.rename(
    columns={col: f"away_{col.replace('team_avg_', '').replace('_game_last15', '_last15')}"
             for col in rolling_cols}
)

savant_away = savant_away.rename(columns={
    "team_name": "away_team_name"
})

Revisar primary key

In [ ]:
df_stats = df_stats.rename(columns={"gamePk": "game_pk"})

In [ ]:
df_stats.columns[df_stats.columns.str.contains("game", case=False)]

Index(['gameNumber', 'gamesInSeries', 'seriesGameNumber', 'game_pk',
       'home_games_played', 'away_games_played'],
      dtype='object')

Merge con métricas del local

In [ ]:
df_model = df_stats.merge(
    savant_home,
    on=["game_pk", "home_team_name"],
    how="left"
)

Merge con métricas del visitante

In [ ]:
df_model = df_model.merge(
    savant_away,
    on=["game_pk", "away_team_name"],
    how="left"
)

Validar el Merge

In [ ]:
print(df_stats.shape)
print(df_model.shape)
#El número de filas debería ser igual.

(8309, 22)
(8309, 44)


In [ ]:
home_cols = [c for c in df_model.columns if c.startswith("home_") and c.endswith("_last15")]
away_cols = [c for c in df_model.columns if c.startswith("away_") and c.endswith("_last15")]

df_model[home_cols + away_cols].isna().mean().sort_values(ascending=False)

,0
home_swing_length_last15,0.178361
home_bat_speed_last15,0.178361
away_bat_speed_last15,0.178361
away_swing_length_last15,0.178361
away_xba_last15,0.012276
away_hit_distance_last15,0.012276
away_woba_last15,0.012276
away_xslg_last15,0.012276
away_iso_last15,0.012276
away_xwoba_last15,0.012276


In [ ]:
df_model["xwoba_diff"] = df_model["home_xwoba_last15"] - df_model["away_xwoba_last15"]
df_model["xba_diff"] = df_model["home_xba_last15"] - df_model["away_xba_last15"]
df_model["launch_speed_diff"] = df_model["home_launch_speed_last15"] - df_model["away_launch_speed_last15"]
df_model["iso_diff"] = df_model["home_iso_last15"] - df_model["away_iso_last15"]

Validacion Final

In [ ]:
savant_cols = [
    c for c in df_model.columns
    if c.startswith("home_") or c.startswith("away_")
]

print(len(savant_cols))
print(sorted(savant_cols))

33
['away_babip_last15', 'away_bat_speed_last15', 'away_games_played', 'away_hit_distance_last15', 'away_iso_last15', 'away_launch_angle_last15', 'away_launch_speed_last15', 'away_losses_before', 'away_pct_before', 'away_swing_length_last15', 'away_team_name', 'away_wins_before', 'away_woba_last15', 'away_xba_last15', 'away_xslg_last15', 'away_xwoba_last15', 'home_babip_last15', 'home_bat_speed_last15', 'home_games_played', 'home_hit_distance_last15', 'home_iso_last15', 'home_launch_angle_last15', 'home_launch_speed_last15', 'home_losses_before', 'home_pct_before', 'home_swing_length_last15', 'home_team_name', 'home_win', 'home_wins_before', 'home_woba_last15', 'home_xba_last15', 'home_xslg_last15', 'home_xwoba_last15']


In [ ]:
(
    df_model[savant_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

,0
away_bat_speed_last15,0.178361
home_swing_length_last15,0.178361
away_swing_length_last15,0.178361
home_bat_speed_last15,0.178361
away_xwoba_last15,0.012276
away_launch_angle_last15,0.012276
away_launch_speed_last15,0.012276
away_xba_last15,0.012276
away_babip_last15,0.012276
away_woba_last15,0.012276


In [ ]:
df_model[
    [
        "home_team_name",
        "away_team_name",
        "home_xwoba_last15",
        "away_xwoba_last15"
    ]
].head(20)

,home_team_name,away_team_name,home_xwoba_last15,away_xwoba_last15
0,Washington Nationals,Atlanta Braves,NaN,NaN
1,New York Yankees,San Francisco Giants,NaN,NaN
2,Boston Red Sox,Baltimore Orioles,NaN,NaN
3,Chicago Cubs,Milwaukee Brewers,NaN,NaN
4,Tampa Bay Rays,Detroit Tigers,NaN,NaN
5,Texas Rangers,Philadelphia Phillies,NaN,NaN
6,Cincinnati Reds,Pittsburgh Pirates,NaN,NaN
7,St. Louis Cardinals,Toronto Blue Jays,NaN,NaN
8,Kansas City Royals,Minnesota Twins,NaN,NaN
9,Miami Marlins,New York Mets,NaN,NaN


In [ ]:
df_model["home_xwoba_last15"].notna().mean()

np.float64(0.98796485738356)

In [ ]:
df_model["away_xwoba_last15"].notna().mean()

np.float64(0.9877241545312312)

Crear diferenciales

In [ ]:
df_model["xwoba_diff"] = (
    df_model["home_xwoba_last15"]
    - df_model["away_xwoba_last15"]
)

df_model["xba_diff"] = (
    df_model["home_xba_last15"]
    - df_model["away_xba_last15"]
)

df_model["launch_speed_diff"] = (
    df_model["home_launch_speed_last15"]
    - df_model["away_launch_speed_last15"]
)

df_model["iso_diff"] = (
    df_model["home_iso_last15"]
    - df_model["away_iso_last15"]
)

Guardar DF_Model en Parquet

In [ ]:
print(df_model.shape)

df_model.info()

(8309, 48)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8309 entries, 0 to 8308
Data columns (total 48 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   dayNight                  8309 non-null   object 
 1   gameNumber                8309 non-null   int64  
 2   scheduledInnings          8309 non-null   int64  
 3   gamesInSeries             8309 non-null   float64
 4   seriesGameNumber          8309 non-null   float64
 5   teams.away.team.name      8309 non-null   object 
 6   away_wins_before          8309 non-null   int64  
 7   away_losses_before        8309 non-null   int64  
 8   away_pct_before           8309 non-null   float64
 9   teams.home.team.name      8309 non-null   object 
 10  home_wins_before          8309 non-null   int64  
 11  home_losses_before        8309 non-null   int64  
 12  home_pct_before           8309 non-null   float64
 13  venue.name                8309 non-null   object 
 1

In [ ]:
df_model.to_parquet("df_model.parquet", index=False)